# 模型归因分析 v1

目标：验证 `src.analysis.explain_model_artifact` 的稳定归因接口。覆盖 native linear、native boosting、proxy graph 三类模型，输出 feature attribution sidecar 和中文结论。

In [1]:
from __future__ import annotations

import sys
from pathlib import Path

ROOT = None
for candidate in [Path.cwd(), *Path.cwd().parents]:
    if (candidate / "src").exists() and (candidate / "SPEC.md").exists():
        ROOT = candidate
        break
if ROOT is None:
    raise RuntimeError("cannot locate repo root")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

CARD = ROOT / "research" / "model_attribution_analysis_v1"
CARD.mkdir(parents=True, exist_ok=True)

START = "2026-01-01"
END = "2026-06-09"
TRAIN_END = "2026-04-30"
MODEL_NAMES = ("ridge_mse", "xgboost_mse", "gcn_mse")
CARD

PosixPath('/Users/rziiiii/dev/workspace/research/model_attribution_analysis_v1')

In [2]:
from src.analysis import write_model_attribution_card
from src.data import build_supervised_dataset, load_soxl_support_frame

price_frame = load_soxl_support_frame(start=START, end=END)
dataset = build_supervised_dataset(price_frame, horizon=1)
train = dataset.frame.loc[dataset.frame["date"] <= TRAIN_END].copy()
test = dataset.frame.loc[dataset.frame["date"] > TRAIN_END].copy()
feature_columns = dataset.feature_columns[:8]

assert not train.empty and not test.empty
train.shape, test.shape, feature_columns

((82, 18),
 (26, 18),
 ('soxl_return_1d',
  'soxx_return_1d',
  'soxl_return_5d',
  'soxx_return_5d',
  'soxl_soxx_relative_5d',
  'vix_level',
  'vix_change_1d',
  'intraday_range'))

In [3]:
result = write_model_attribution_card(
    CARD,
    model_names=MODEL_NAMES,
    train=train,
    test=test,
    feature_columns=feature_columns,
    top_n=6,
)

feature_attribution = result.feature_attribution
top_by_model = result.top_features_by_model
capabilities = result.model_capabilities
analysis_summary = result.summary

assert set(feature_attribution["model"]) == set(MODEL_NAMES)
assert {"permutation", "sensitivity"}.issubset(set(feature_attribution["attribution_kind"]))
assert "coefficient" in set(feature_attribution.loc[feature_attribution["model"].eq("ridge_mse"), "attribution_kind"])
feature_attribution.head(12)

,model,feature,attribution_kind,importance,normalized_importance,rank,direction
0,ridge_mse,soxx_return_5d,coefficient,0.032091,0.310861,1,-1.0
1,ridge_mse,soxl_soxx_relative_5d,coefficient,0.030108,0.291646,2,1.0
2,ridge_mse,soxx_return_1d,coefficient,0.013650,0.132224,3,1.0
3,ridge_mse,vix_change_1d,coefficient,0.010937,0.105945,4,1.0
4,ridge_mse,soxl_return_5d,coefficient,0.010432,0.101053,5,1.0
5,ridge_mse,vix_level,coefficient,0.006015,0.058271,6,1.0
6,ridge_mse,soxl_soxx_relative_5d,permutation,0.043800,0.296345,1,1.0
7,ridge_mse,soxx_return_5d,permutation,0.041476,0.280621,2,1.0
8,ridge_mse,soxx_return_1d,permutation,0.027973,0.189260,3,-1.0
9,ridge_mse,soxl_return_5d,permutation,0.014461,0.097840,4,-1.0


In [4]:
assert analysis_summary["schema_version"] == "analysis.model_attribution.batch.v1"
assert analysis_summary["models"] == list(MODEL_NAMES)
assert analysis_summary["proxy_models"] == ["gcn_mse"]
assert top_by_model.groupby("model").size().max() == 6
assert (CARD / "analysis_summary.json").exists()
assert (CARD / "feature_attribution.csv").exists()
assert (CARD / "top_features_by_model.csv").exists()

{
    "summary": analysis_summary,
    "top_features": top_by_model.head(9).to_dict(orient="records"),
}

{'summary': {'status': 'completed',
  'schema_version': 'analysis.model_attribution.batch.v1',
  'models': ['ridge_mse', 'xgboost_mse', 'gcn_mse'],
  'model_count': 3,
  'train_start': '2026-01-02',
  'train_end': '2026-04-30',
  'test_start': '2026-05-01',
  'test_end': '2026-06-08',
  'feature_count': 8,
  'top_n': 6,
  'attribution_kinds': {'coefficient': 6,
   'feature_importance': 6,
   'permutation': 18,
   'sensitivity': 18},
  'proxy_models': ['gcn_mse'],
  'artifacts': {'feature_attribution': 'feature_attribution.csv',
   'model_attribution_summaries': 'model_attribution_summaries.json',
   'model_capabilities': 'model_capabilities.csv',
   'top_features_by_model': 'top_features_by_model.csv'},
  'risk_notes': ['Permutation 和 sensitivity 是局部解释工具，不等价于因果解释。',
   'proxy_feature_adapter 模型的归因解释的是当前 proxy feature path，不等价于完整论文结构。',
   '短测试窗归因只适合 review，应继续在 walk-forward 和 regime 分段中复验。']},
 'top_features': [{'model': 'gcn_mse',
   'feature': 'soxx_return_5d',
   'normalized_importa

## 结论

- 本次在 `ridge_mse`、`xgboost_mse`、`gcn_mse` 上验证了 `src.analysis.write_model_attribution_card` 批量归因接口；输出包含 `feature_attribution.csv`、`top_features_by_model.csv`、`model_capabilities.csv`、`model_attribution_summaries.json` 和 `analysis_summary.json`。
- `analysis_summary.json` 已使用稳定 schema `analysis.model_attribution.batch.v1`，notebook 不再手写业务 sidecar 逻辑，只负责调用 src 接口和记录中文结论。
- `ridge_mse` 同时具备 coefficient、permutation、sensitivity 三类归因；`xgboost_mse` 和 `gcn_mse` 至少具备 permutation 与 sensitivity，满足 tabular/native 与 proxy graph 的共同解释入口。
- `gcn_mse` 当前是 `proxy_feature_adapter`，归因结果只能解释现有 proxy feature path，不能解释完整论文 GCN 结构。
- 不晋升 production：本次只验证分析接口和一个短测试窗。下一步应把 attribution sidecar 接入 dashboard manifest，并在 walk-forward/regime 分段中比较 top feature 稳定性。